In [1]:
import numpy as np

In [2]:
import os
import sys
import pathlib
from pathlib import Path

In [3]:
prj_root = Path("__file__").absolute().parent.parent
if (t := str(prj_root)) not in sys.path:
    sys.path.append(t)

In [4]:
from util.obo_parser import Ontology, WangGOSim

In [15]:
from goatools.obo_parser import GODag
from tqdm import tqdm

In [6]:
go_path = "/data0/shaojiangyi/pprogo-flg-2/data/data-netgo/go.obo"
ont_obj = Ontology(go_path, with_rels=True)

In [7]:
go_dag = GODag(go_path)

/data0/shaojiangyi/pprogo-flg-2/data/data-netgo/go.obo: fmt(1.2) rel(2021-01-01) 47,285 Terms


In [8]:
union_term_paths = [
    "/data0/shaojiangyi/pprogo-flg-2/results/union_space_preds_only1/cc/test/union_go_terms.npy",
    "/data0/shaojiangyi/pprogo-flg-2/results/union_space_preds_only1/mf/test/union_go_terms.npy",
    "/data0/shaojiangyi/pprogo-flg-2/results/union_space_preds_only1/bp/test/union_go_terms.npy",
]

In [9]:
union_term_lst = [np.load(p, allow_pickle=True) for p in union_term_paths]

In [10]:
union_termset = [set(m.tolist()) for m in union_term_lst]

In [11]:
go_parents_lst = [{x: ont_obj.get_anchestors(x).intersection(union_termset[i])
                   for x in xs.tolist()}
                   for i, xs in enumerate(union_term_lst)]

In [13]:
wang_sim = WangGOSim(go_dag, go_parents_lst[2])

In [16]:
def wang_similarity_matrix(wang_sim: WangGOSim, go_list):
    n = len(go_list)
    mat = np.zeros((n, n), dtype=float)
    for i in tqdm(range(n)):
        mat[i, i] = 1.0
        for j in range(i + 1, n):
            sim = wang_sim.get_sim(go_list[i], go_list[j])
            if sim is None:
                sim = 0.0
            mat[i, j] = mat[j, i] = sim
    return mat

In [17]:
wang_sim_lst = []
for i, go_arr in enumerate(union_term_lst):
    go_names = go_arr.tolist()
    m = wang_similarity_matrix(wang_sim, go_names)
    wang_sim_lst.append(m)

100%|██████████| 21822/21822 [08:37<00:00, 42.16it/s] 


In [19]:
ns = ["cc", "mf", "bp"]

In [20]:
saving_dir = Path("/data0/shaojiangyi/pprogo-flg-2/data")
for i, n in enumerate(ns):
    saving_path = saving_dir / f"{n}" / "hgat_esm2_inductive_features" / "union_go_wangsim.npy"
    np.save(saving_path, wang_sim_lst[i].astype(np.float32))